# Image occurrence export

Image metadata and occurence data are exported from the BioCosmos database to a CSV file or a compressed CSV file for Handika et al. BioCosmos manuscript supplementary materials.
See [definitions and prerequisites](../README.md). Stop the backend first.

In [1]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "backend/app/configs/config.yaml").is_file()
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import json

import pandas as pd
from analyses.helpers.occurrences import COLUMNS, column_dictionary, export_occurrences
from analyses.helpers.publication import load_settings

settings = load_settings(ROOT)
pd.set_option("display.max_colwidth", 120)

## Columns

In [3]:
column_dictionary(COLUMNS)

,column,description
0,specimenID,Occurrence identifier of the specimen (GBIF occurrenceID or the aggregator's record ID). The dorsal and ventral imag...
1,species,Accepted species in the Catalogue of Life. Subspecies are grouped under their species. Empty when the name matched o...
2,sourceDB,"Aggregator the image came from (gbif, scanbugs, ecdysis); records carried by several aggregators keep a combined key..."
3,sex,"Sex as recorded, lowercased."
4,institution,"Institution holding the specimen, by its GRSciColl or publisher name; the institution code when no name was resolved."
5,URI,URI of the original image file.
6,imageID,BioCosmos image identifier that keys the image embeddings and the website.
7,view,"Dorsal or ventral, predicted by the view classifier in Lürig et al., 2025: https://doi.org/10.1111/ele.14495."
8,viewConfidence,Classifier probability of the view.
9,imageWidth,Width in pixels of the segmented specimen.


## Export

In [4]:
paths = export_occurrences(settings, COLUMNS)
manifest = json.loads(paths["manifest"].read_text())

print(
    f"{manifest['rows']:,} images of {manifest['specimens']:,} specimens "
    f"and {manifest['species']:,} accepted species"
)
print(
    f"Catalogue of Life {manifest['catalogue_of_life']['version']}: "
    f"{manifest['catalogue_of_life']['url']}"
)
print(
    f"GADM {manifest['gadm']['version']} ({manifest['gadm']['layer']}): {manifest['gadm']['url']}"
)
pd.DataFrame(manifest["files"]).T

619,739 images of 515,141 specimens and 8,401 accepted species
Catalogue of Life 2026-10-16 XR: https://doi.org/10.48580/dgzlf
GADM 4.1 (ADM_1): https://gadm.org/download_world.html


,bytes,sha256
image_occurrences.parquet,40156511,52882cd07e8ae8cad77509b0d3c6be41686c89a984c41f0d721d94965c840e6d
image_occurrences.csv.gz,51907768,5da866cf3fe9130bf4381cab9aad68271eb0e69527a6127b086b6db8ef6435a5
image_occurrences_columns.csv,2468,708f83f348776f3eadfca470015a3f253f7aaa81f808155e2eb96dc23829076b


## Check

In [5]:
occurrences = pd.read_parquet(paths["parquet"])

occurrences.head()

,specimenID,species,sourceDB,sex,institution,URI,imageID,view,viewConfidence,imageWidth,...,country,stateProvince,county,locality,decimalLatitude,decimalLongitude,coordinateValidation,eventDate,year,recordedBy
0,019ca92a-23ca-4dea-a362-7ebfd1f1d97c,Abaeis nicippe,gbif,NaN,"San Diego Natural History Museum, theNAT",https://ecdysis.org/imglib/scan/misc/202211/SDNHM266016.JPG,d9860cbc-cb366003-59067992-7910f8d4_1,dorsal,0.999999,942,...,United States,California,San Diego,"Ribbonwood Road, Big Country Ranch",NaN,NaN,MISSING_COORDINATE,1998-08-05,1998.0,D.K. Faulkner
1,03d5a6c2-3f61-4495-9dfd-54cff478ca15,Abaeis nicippe,gbif,NaN,"San Diego Natural History Museum, theNAT",https://ecdysis.org/imglib/scan/misc/202211/SDNHM266015.JPG,1c6b9acd-550c6ce2-2c0db3f4-26ac73b2_1,dorsal,0.999992,891,...,United States,Arizona,Santa Cruz,Madera Canyon,NaN,NaN,MISSING_COORDINATE,1959-09-06,1959.0,W.S. Wright
2,0ac58b3e-75c7-4691-bf57-cd26281d7148,Abaeis nicippe,gbif,NaN,"University of Nevada, Museum of Biology",https://api.idigbio.org/v2/media/98bc558b526e28ca1cd3aa7f50c80b34?size=fullsize,98bc558b-526e28ca-1cd3aa7f-50c80b34_1,ventral,1.000000,3115,...,United States,Georgia,Lee Co.,"6 Miles West of Albany on Highway #82, then 0.7 Miles North, 200ft",31.578467,-84.257963,VALID,1978-07-08,1978.0,Fred Ryser
3,0bbb28ca-aa60-4da3-bb24-0a817a6a5fb0,Abaeis nicippe,gbif,male,"San Diego Natural History Museum, theNAT",https://ecdysis.org/imglib/scan/misc/202211/SDNHM266022.JPG,6ebd8a81-f01c935b-82626162-a17192e1_1,dorsal,1.000000,649,...,United States,Arizona,Pima,"30 miles southeast of Tucson, Empire Mountains",NaN,NaN,MISSING_COORDINATE,1974-09-10,1974.0,J.D. Lang
4,10836f09-74d8-4e11-a88b-a4b2dfde75c0,Abaeis nicippe,gbif,NaN,"San Diego Natural History Museum, theNAT",https://ecdysis.org/imglib/scan/misc/202211/SDNHM266019.JPG,49fe8f87-4ab541c6-b2172083-8901bf5b_1,dorsal,0.999999,984,...,Mexico,Baja California Sur,NaN,"Sierra de Guadalupe, San Sebastion",NaN,NaN,MISSING_COORDINATE,1997-10-25,1997.0,D.K. Faulkner


In [6]:
# Share of images with each column filled.
occurrences.notna().mean().mul(100).round(1).rename("filled_%").to_frame()

,filled_%
specimenID,100.0
species,99.2
sourceDB,100.0
sex,19.0
institution,91.1
URI,100.0
imageID,100.0
view,100.0
viewConfidence,100.0
imageWidth,100.0
